# 3 · Optimization

**What you will do.** Search the design space for the best designs: first with EnergyPlus alone (slow), then with the surrogate (fast), and finally check the best design with a real simulation.

**You need.** Notebook 1 (vocabulary). Part B needs the simulations and the surrogate of notebook 2: they are stored in this folder (`surrogate_simple_simulation.csv`, `best_surrogate.pkl`); if the surrogate file is missing it is trained again.

**Run time.** about 8 minutes in all (part A runs about 30 simulations). With `FULL_STUDY = True` part A runs 200: plan for about 45 minutes (a simulation takes about 14 seconds when two run at a time).

The four notebooks of this series use the same building model (`test.idf`, a small house) and the same weather file
(Turin). Work in order: 1 *bdopt overview* → 2 *surrogate modelling* → 3 *optimization* → 4 *adaptive sampling*. Cells
marked **Your turn** are exercises: change something and observe what happens.

## Multi-objective optimisation

Designs usually have to satisfy conflicting objectives: for example less heating demand tends to mean more cooling demand
in summer. There is no single best design but a set of **non-dominated** designs, the *Pareto front*: none of them is
better than another on every objective at once. We find it with a genetic algorithm, NSGA-II.

**Part A** runs the optimiser directly on EnergyPlus: every design the algorithm tries is simulated.
**Part B** trains nothing new (unless the saved surrogate is missing): it loads the surrogate of notebook 2 and lets the algorithm try thousands of designs, then verifies the best one with EnergyPlus.
Part B does not depend on part A: you can start from the heading *Part B*.

# Part A · optimisation with EnergyPlus

### Import libraries

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from bdopt import (EnergyPlus, EnergyPlusEvaluator, FieldSelector, FunctionEvaluator, IntegerParameter,
                   Parameter, Problem, RangeParameter, load_idf, sampling, viz)
from bdopt.optimize import nsga2
from bdopt.surrogate import MLPSurrogate, Surrogate
from window_to_wall import wwr      # the window-to-wall ratio as a design variable: see 1_bdopt_overview.ipynb

FULL_STUDY = False      # False: a small population and few generations (about 30 simulations); True: 200 simulations (~45 min)

### Load the base EnergyPlus .idf file

In [ ]:
idf = load_idf("test.idf")
epw_file = "ITA_Torino.160590_IWEC.epw"

### Define design parameters and ranges
Define a parameter list, in this case the thickness of an insulation layer and the window-to-wall ratio: a field of a material (`FieldSelector`) and a geometry edit (`wwr`, a `GenericSelector` in `window_to_wall.py`).

In [ ]:
parameters = [
    Parameter(FieldSelector("Material", "XPS Extruded Polystyrene  - CO2 Blowing_0041", "Thickness"),
              RangeParameter(0.001, 0.3), name="Insulation Thickness"),      # [m]
    wwr(0.1, 0.8),                                                           # window-to-wall ratio between 0.1 and 0.8
]

### Objectives
Using Heating and Cooling energy outputs as simulation objectives, make a problem instance from these parameters and objectives.

In [ ]:
objectives = ["DistrictCooling:Facility", "DistrictHeating:Facility"]
problem = Problem(parameters, objectives)

### Set up the EnergyPlus evaluator object to run simulations for this building and problem

In [ ]:
evaluator = EnergyPlusEvaluator(
    problem, idf, epw_file,
    out_dir="/tmp/bdopt_runs",   # work files go to the container's own disk (the shared folder is much slower)
    err_dir="outputdir",         # runs that fail are kept here, with the EnergyPlus error file
    processes=2,                 # EnergyPlus runs in parallel
)
runs = pd.DataFrame({"Insulation Thickness": [0.2], "Window-to-wall ratio": [0.5]})   # one design: 0.2 m of insulation and 50% glazing

In [ ]:
runs

In [ ]:
outputs = evaluator.evaluate(runs)  # Run this as a test: one full-year EnergyPlus simulation
outputs

## Run the Genetic Algorithm
`nsga2` takes the problem and *what evaluates it*: here the EnergyPlus evaluator, so every design the algorithm tries is simulated (a full year each; each generation runs `pop_size` simulations, in parallel, and a design already simulated is taken from the cache).

To keep the lecture short we use a population of 10 designs for 3 generations (about 30 simulations); a real study needs much more (`FULL_STUDY = True`). To plot every design that was tried, and not only the best ones, the evaluator is wrapped in a function that records what it computes: any Python function can play the role of an evaluator.

In [ ]:
tried = []      # every design the algorithm simulates, with its results

def simulate_and_record(designs):
    results = evaluator.evaluate(designs, keep_input=True)
    tried.append(results)
    return results[objectives].to_numpy()


recording_evaluator = FunctionEvaluator(problem, simulate_and_record, vectorized=True)

In [ ]:
POP_SIZE, N_GEN = (20, 10) if FULL_STUDY else (10, 3)
front = nsga2(problem, recording_evaluator, pop_size=POP_SIZE, n_gen=N_GEN, seed=0)
results = pd.concat(tried, ignore_index=True).drop_duplicates()
front.df

### Visualize the results

In [ ]:
optres = front.df     # the non-dominated designs
plt.plot(
    results["DistrictCooling:Facility"], results["DistrictHeating:Facility"], "x"
)  # Plot all results in the background as blue crosses
plt.plot(
    optres["DistrictCooling:Facility"], optres["DistrictHeating:Facility"], "ro"
)  # Plot optimal results in red
plt.xlabel("Cooling demand")
plt.ylabel("Heating demand")

In [ ]:
optres = optres.sort_values("DistrictHeating:Facility")  # Sort by the first objective
ax = optres.plot.bar(
    subplots=True, legend=None, figsize=(10, 10)
)  # Plot the variable values and the objectives of each of the optimal solutions

**Your turn.** Read the plot: moving along the front from the left to the right, what do you gain and what do you lose? Which design would you build, and why?

### Giving constraints to the optimization

bdopt constrains the *design variables*: `Problem(parameters, objectives, constraints=[g])`, where `g(design)` receives a design (a dictionary parameter name -> value) and returns a number; the design is feasible when the number is `<= 0`. Samplers only draw feasible designs and the optimisers respect the constraint. There is an example at the end of part B.

# Part B · optimisation with the surrogate

Simulating every design the algorithm tries is slow. A surrogate model trained on a few simulations answers in milliseconds, so the algorithm can try thousands of designs. Notebook 2 produced two files: the simulations (`surrogate_simple_simulation.csv`) and the trained surrogate (`best_surrogate.pkl`).

In [ ]:
# the simulations the surrogate was trained on (stored in this folder, written by 2_surrogate_modelling.ipynb)
import os

if not os.path.exists('surrogate_simple_simulation.csv'):
    raise FileNotFoundError(
        f"surrogate_simple_simulation.csv not found in {os.getcwd()}. It is written by 2_surrogate_modelling.ipynb: "
        "open it and run it from the top to the end, then come back here.")
samples = pd.read_csv('surrogate_simple_simulation.csv', index_col=0)
samples

The problem is the one of `2_surrogate_modelling.ipynb`: five design variables and three objectives.

In [ ]:
parameters = [
    Parameter(FieldSelector('Material', 'XPS Extruded Polystyrene  - CO2 Blowing_0041', 'Thickness'),
              RangeParameter(0.01, 0.45), name='Insulation Thickness'),
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Conductivity'),
              RangeParameter(0.1, 1), name='Glazing conductivity'),
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Solar Transmittance at Normal Incidence'),
              RangeParameter(0.01, 0.8), name='Glazing solar transmittance'),
    wwr(0.1, 0.8),
    Parameter(FieldSelector('Lights', '*', 'Watts per Zone Floor Area'), IntegerParameter(8, 12), name='Lights Watts/Area'),
]
objectives = ['Electricity:Facility', 'DistrictHeating:Facility', 'DistrictCooling:Facility']   # EnergyPlus meters [J]
problem = Problem(parameters, objectives)

features = problem.parameter_names

# the table must have the columns of this problem
absent = [n for n in features + objectives if n not in samples.columns]
if absent:
    raise ValueError(
        f"surrogate_simple_simulation.csv has the columns {list(samples.columns)}, but this problem needs {absent} as well. "
        "The file was written for another problem: run 2_surrogate_modelling.ipynb again from the top to the end "
        "(with RESIMULATE = True), and come back here.")
inputs = samples[features]
outputs = samples[objectives]
inputs.head()

Load the surrogate. bdopt surrogates scale their inputs and outputs inside, so no scaler has to be rebuilt. If the file is missing, or does not match this problem, the surrogate is trained again on the stored simulations (about a minute).

In [ ]:
try:
    surrogate = Surrogate.load('best_surrogate.pkl', problem)
    predicted = surrogate.predict(inputs)
    # the surrogate was trained on these very simulations, so it must be close to them (a file from another problem, or an
    # old scikit-learn pickle, loads without complaint and gives numbers that mean nothing)
    error = np.abs((np.asarray(predicted) - outputs.to_numpy()) / outputs.to_numpy()).mean()
    if not error < 0.5:
        raise ValueError(f"on the simulations it was trained on the surrogate is off by {100 * error:.0f}% on average")
except Exception as e:
    print(f"best_surrogate.pkl could not be used ({type(e).__name__}: {e}); training the surrogate on the stored simulations instead.")
    surrogate = MLPSurrogate(random_state=0).fit(samples, problem)
surrogate.predict(inputs.head(3))       # what the surrogate says about the first three simulated designs

In [ ]:
outputs.head(3)                         # what EnergyPlus said

In [ ]:
# running the NSGA-II optimizer on the surrogate: thousands of evaluations in seconds
front = nsga2(problem, surrogate, pop_size=100, n_gen=100, seed=0)

# computing Q: energy per square metre of floor area
HOUSE_DIMENSION = 118       # [m2]
results = front.df.copy()
results['Q'] = (results['Electricity:Facility'] + results['DistrictHeating:Facility'] + results['DistrictCooling:Facility']) / HOUSE_DIMENSION
results

In [ ]:
# plotting results: random designs evaluated by the surrogate in the background, the non-dominated designs in red
background = sampling.lhs(problem, 500, seed=1)
background = background.join(surrogate.predict(background))
viz.pareto(front, background=background)
plt.show()

In [ ]:
df = front.df.copy()
df['Total'] = df['Electricity:Facility'] + df['DistrictHeating:Facility'] + df['DistrictCooling:Facility']
df['Dist'] = np.sqrt(df['Electricity:Facility'] ** 2 + df['DistrictHeating:Facility'] ** 2 + df['DistrictCooling:Facility'] ** 2)

df[df.Dist == df.Dist.min()]

In [ ]:
viz.pairplot(front.df, problem, size=3)
plt.show()

In [ ]:
viz.correlation_heatmap(front.df, problem)
plt.show()

bdopt can also pick the design of the front closest to the ideal point, after scaling every objective to 0-1 over the front (so that an objective in joules does not weigh more only because of its numbers):

In [ ]:
front.closest_to_ideal()

### Is the surrogate right? Check the chosen design with EnergyPlus

The optimiser trusted the surrogate. Before building anything, write the chosen design into the model, save it, and simulate it: the surrogate's prediction and the simulation should agree.

In [ ]:
best = front.closest_to_ideal()
design = {name: float(best[name]) for name in problem.parameter_names}
design

In [ ]:
idd = EnergyPlus.find().idd
optimal = idf.copy()
problem.apply(optimal, design, idd)         # writes the design into the model, with the same selectors used to simulate
optimal.save('optimal_design.idf')

In [ ]:
check = EnergyPlusEvaluator(problem, idf, epw_file, out_dir='/tmp/bdopt_runs', err_dir='outputdir', processes=1)
simulated = check.evaluate(pd.DataFrame([design]))[objectives].iloc[0]
predicted = surrogate.predict(pd.DataFrame([design])).iloc[0]
pd.DataFrame({'surrogate [kWh]': predicted / 3.6e6, 'EnergyPlus [kWh]': simulated / 3.6e6,
              'difference [%]': 100 * (predicted - simulated) / simulated}).round(1)

**Your turn.** How far is the surrogate from the simulation at the optimum? Optimisers tend to go where the surrogate is *least* accurate. What would you do to improve it (more samples near the front, a different model)?

### Choose a design yourself

No design of the front is the best: it depends on how much each objective matters to *you*. Move the sliders (0 = it does not matter, 1 = it matters fully): the star marks the design that `front.closest_to_ideal(weights)` picks, in the three pairs of objectives. Under the plots: that design, and what the surrogate predicts for it. The button simulates it with EnergyPlus (about half a minute) and compares.

In [ ]:
import explore       # explore.py, next to this notebook

chooser = explore.ParetoExplorer(front, problem, surrogate, evaluator=check, background=background)
chooser.widget

**Your turn.** Give all the weight to the electricity: which design do you get, and what do you pay in heating and cooling? Then do the opposite. `chooser.design` is the design on the screen.

### Which parameter values go with the good designs?

The same brushable plot as in notebook 2, now on the designs of the front: drag on the *DistrictHeating* axis to keep the designs with the lowest heating demand and read the parameters on the left.

In [ ]:
explore.parallel_coordinates(front.df, problem).show()

### A constraint on a design variable

Suppose the insulation layer cannot be thicker than 0.25 m. The constraint is a function of the design: feasible when it returns a number `<= 0`.

In [ ]:
def thin_insulation(design):
    return design['Insulation Thickness'] - 0.25


constrained_problem = Problem(parameters, objectives, constraints=[thin_insulation])
front_constrained = nsga2(constrained_problem, surrogate, pop_size=100, n_gen=100, seed=0)

print('thickest insulation, without the constraint:', round(front.df['Insulation Thickness'].max(), 3), 'm')
print('thickest insulation, with the constraint:   ', round(front_constrained.df['Insulation Thickness'].max(), 3), 'm')

In [ ]:
plt.plot(front.df["DistrictCooling:Facility"], front.df["DistrictHeating:Facility"], "bo", label="no constraint")
plt.plot(front_constrained.df["DistrictCooling:Facility"], front_constrained.df["DistrictHeating:Facility"], "ro",
         label="insulation <= 0.25 m")
plt.xlabel("Cooling demand")
plt.ylabel("Heating demand")
plt.legend()
plt.show()

## Summary
- A multi-objective optimisation returns a **Pareto front**, not one design: the choice among its designs is yours.
- With a **surrogate** the algorithm can try thousands of designs; always **verify** the chosen design with a real simulation.
- bdopt constrains **design variables**: `Problem(..., constraints=[g])`.

Next: **4_adaptive_sampling.ipynb** lets the sampler choose which simulations to run.

## Sources and credits

**Origin of this series.** The four notebooks are adapted from the notebooks of the course, which were
written for the toolbox BESOS (Westermann, P., Christiaanse, T.V., Beckett, W., Kovacs, P., Evins, R. (2021). besos: Building and Energy Simulation, Optimization and Surrogate Modelling. *Journal of Open Source Software* 6(60), 2677. https://doi.org/10.21105/joss.02677; code and examples under GPL-3.0, https://gitlab.com/energyincities/besos). They were
rewritten in October 2026 for **bdopt** (MIT licence, https://github.com/LorenzoBottaccioli/bdopt) and extended; the figures marked
above come from those notebooks or from the BESOS documentation.

**Model and weather.** `test.idf` was generated with DesignBuilder for the course. `ITA_Torino.160590_IWEC.epw`: IWEC data,
original source data (c) 2001 American Society of Heating, Refrigerating and Air-Conditioning Engineers (ASHRAE), Inc., Atlanta,
GA, USA, converted to EPW for EnergyPlus; licensed for your own use, not to be passed to third parties without ASHRAE's permission.

**Simulation.** The building is simulated with EnergyPlus version 9.6 (U.S. Department of Energy and partners, BSD-style licence).

**References for this notebook.**
- Crawley, D.B., Lawrie, L.K., Winkelmann, F.C., Buhl, W.F., Huang, Y.J., Pedersen, C.O., Strand, R.K., Liesen, R.J., Fisher, D.E., Witte, M.J., Glazer, J. (2001). EnergyPlus: creating a new-generation building energy simulation program. *Energy and Buildings* 33(4), 319-331. https://doi.org/10.1016/S0378-7788(00)00114-6
- Deb, K., Pratap, A., Agarwal, S., Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm: NSGA-II. *IEEE Transactions on Evolutionary Computation* 6(2), 182-197. https://doi.org/10.1109/4235.996017
- Blank, J., Deb, K. (2020). pymoo: multi-objective optimization in Python. *IEEE Access* 8, 89497-89509. https://doi.org/10.1109/ACCESS.2020.2990567

All of the platform's credits and licences are listed on the page **Credits & licences** of the platform.